# Customer Support Analytics
**A practical ticket analytics project covering service levels, resolution performance, repeat contacts and backlog.**

The notebook generates a synthetic support-ticket dataset and builds a compact dashboard using pandas, matplotlib and seaborn. All records and results are illustrative.

In [ ]:
# If needed:
# %pip install pandas numpy matplotlib seaborn jupyter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

rng = np.random.default_rng(21)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Generate sample customer-query data

In [ ]:
n = 1600
created = pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D")
categories = rng.choice(["Account Access","Billing","Delivery","Product Issue","Returns","General"], n,
                        p=[.18,.20,.18,.19,.12,.13])
channels = rng.choice(["Email","Chat","Phone","Web Form"], n, p=[.32,.34,.22,.12])
priorities = rng.choice(["Low","Medium","High","Urgent"], n, p=[.20,.45,.27,.08])

cat_factor = pd.Series(categories).map({
    "Account Access":1.0,"Billing":1.2,"Delivery":1.5,"Product Issue":2.0,"Returns":1.6,"General":.8
}).to_numpy()
priority_factor = pd.Series(priorities).map({"Low":1.5,"Medium":1.0,"High":.65,"Urgent":.4}).to_numpy()
channel_factor = pd.Series(channels).map({"Email":1.4,"Chat":.45,"Phone":.65,"Web Form":1.8}).to_numpy()

response_hours = np.maximum(.05, rng.gamma(1.8, 2.0, n) * priority_factor * channel_factor)
resolution_hours = np.maximum(.25, rng.gamma(2.0, 8.0, n) * cat_factor * priority_factor)
resolved_flag = rng.random(n) > (.07 + (priorities=="Low")*.02)
cutoff = pd.Timestamp("2026-01-15")
customer_ids = [f"CUST-{x:04d}" for x in rng.integers(1, 501, n)]
ticket_created = pd.Series(created)
response_at = ticket_created + pd.to_timedelta(response_hours, unit="h")
resolved_at = ticket_created + pd.to_timedelta(resolution_hours, unit="h")
resolved_at = pd.Series(resolved_at).where(resolved_flag, pd.NaT)

tickets = pd.DataFrame({
    "ticket_id":[f"TKT-{i:05d}" for i in range(1,n+1)],
    "customer_id":customer_ids,
    "created_at":ticket_created,
    "category":categories,
    "channel":channels,
    "priority":priorities,
    "response_at":response_at,
    "resolved_at":resolved_at
})
tickets["response_hours"] = (tickets["response_at"]-tickets["created_at"]).dt.total_seconds()/3600
tickets["resolution_hours"] = (tickets["resolved_at"]-tickets["created_at"]).dt.total_seconds()/3600
tickets["status"] = np.where(tickets["resolved_at"].notna(), "Resolved", "Unresolved")
tickets["created_month"] = tickets["created_at"].dt.to_period("M").astype(str)

# Repeat contact: same customer submits another ticket in the same category.
tickets = tickets.sort_values(["customer_id","category","created_at"])
tickets["previous_same_issue"] = tickets.groupby(["customer_id","category"]).cumcount()
tickets["repeat_query"] = tickets["previous_same_issue"] > 0
tickets = tickets.sort_values("created_at").reset_index(drop=True)

tickets.head()


## 2. Data checks and service KPIs

In [ ]:
print("Ticket records:", len(tickets))
print("Duplicate ticket IDs:", tickets["ticket_id"].duplicated().sum())
display(tickets.isna().sum().to_frame("missing_values"))

resolved = tickets[tickets["status"].eq("Resolved")]
kpis = pd.Series({
    "Total tickets": len(tickets),
    "Resolved tickets (%)": tickets["status"].eq("Resolved").mean()*100,
    "Unresolved tickets": tickets["status"].eq("Unresolved").sum(),
    "Average first response (hours)": tickets["response_hours"].mean(),
    "Median first response (hours)": tickets["response_hours"].median(),
    "Average resolution (hours, resolved only)": resolved["resolution_hours"].mean(),
    "Repeat query rate (%)": tickets["repeat_query"].mean()*100
})
display(kpis.to_frame("value"))


## 3. Performance by category, channel and priority

In [ ]:
category_perf = tickets.groupby("category").agg(
    tickets=("ticket_id","count"),
    avg_response_hours=("response_hours","mean"),
    avg_resolution_hours=("resolution_hours","mean"),
    unresolved=("status",lambda s:(s=="Unresolved").sum()),
    repeat_rate=("repeat_query","mean")
).sort_values("avg_resolution_hours", ascending=False)
category_perf["repeat_rate"] *= 100
display(category_perf.round(2))

channel_perf = tickets.groupby("channel").agg(
    tickets=("ticket_id","count"),
    avg_response_hours=("response_hours","mean"),
    avg_resolution_hours=("resolution_hours","mean")
).sort_values("avg_response_hours")
display(channel_perf.round(2))

priority_perf = tickets.groupby("priority").agg(
    tickets=("ticket_id","count"),
    avg_response_hours=("response_hours","mean"),
    avg_resolution_hours=("resolution_hours","mean"),
    unresolved=("status",lambda s:(s=="Unresolved").sum())
)
display(priority_perf.round(2))


## 4. Repeat queries and unresolved cases

In [ ]:
repeat_summary = tickets.groupby("category").agg(
    total_tickets=("ticket_id","count"),
    repeat_tickets=("repeat_query","sum"),
    repeat_rate=("repeat_query","mean")
).sort_values("repeat_rate", ascending=False)
repeat_summary["repeat_rate"] *= 100
display(repeat_summary.round(2))

unresolved = tickets[tickets["status"].eq("Unresolved")].copy()
unresolved["ticket_age_days"] = (cutoff - unresolved["created_at"]).dt.days
unresolved_backlog = unresolved.groupby(["priority","category"]).agg(
    open_tickets=("ticket_id","count"),
    avg_age_days=("ticket_age_days","mean")
).sort_values("open_tickets", ascending=False)
display(unresolved_backlog.head(15).round(1))


## 5. Customer support dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

tickets["category"].value_counts().plot(kind="bar", ax=axes[0,0], title="Tickets by category")
axes[0,0].set_ylabel("Ticket count")
axes[0,0].tick_params(axis="x", rotation=35)

tickets.groupby("created_month").size().plot(ax=axes[0,1], marker="o", title="Monthly ticket volume")
axes[0,1].set_ylabel("Ticket count")
axes[0,1].tick_params(axis="x", rotation=60)

sns.barplot(data=tickets, x="priority", y="response_hours",
            order=["Low","Medium","High","Urgent"], estimator="mean", errorbar=None, ax=axes[1,0])
axes[1,0].set_title("Average first response by priority")
axes[1,0].set_ylabel("Hours")

status_counts = tickets["status"].value_counts()
axes[1,1].pie(status_counts.values, labels=status_counts.index, autopct="%1.1f%%", startangle=90)
axes[1,1].set_title("Resolution status mix")

plt.tight_layout()
plt.show()


## 6. SLA-style monitoring (illustrative targets)

In [ ]:
# Example targets only; confirm actual SLA definitions with the business.
response_target = {"Urgent":1, "High":4, "Medium":12, "Low":24}
resolution_target = {"Urgent":8, "High":24, "Medium":72, "Low":120}

tickets["response_target_hours"] = tickets["priority"].map(response_target)
tickets["response_sla_met"] = tickets["response_hours"] <= tickets["response_target_hours"]
tickets["resolution_target_hours"] = tickets["priority"].map(resolution_target)
tickets["resolution_sla_met"] = (
    tickets["resolution_hours"].notna() &
    (tickets["resolution_hours"] <= tickets["resolution_target_hours"])
)

sla = tickets.groupby("priority").agg(
    response_sla_met_pct=("response_sla_met",lambda s:s.mean()*100),
    resolved_ticket_count=("resolution_sla_met","count"),
    resolution_sla_met_pct=("resolution_sla_met",lambda s:s.mean()*100)
)
display(sla.round(2))


## 7. Practical improvement opportunities

In [ ]:
slowest_category = category_perf["avg_resolution_hours"].idxmax()
highest_repeat = repeat_summary["repeat_rate"].idxmax()
slowest_channel = channel_perf["avg_response_hours"].idxmax()
largest_backlog_priority = unresolved["priority"].value_counts().idxmax() if len(unresolved) else "None"

print("Data-led improvement opportunities")
print(f"1. Review knowledge articles, diagnostic steps and escalation paths for {slowest_category}, the slowest-resolving category.")
print(f"2. Analyse root causes and improve self-service for {highest_repeat}, which has the highest repeat-contact rate.")
print(f"3. Review staffing, routing and queue coverage for {slowest_channel}, the slowest first-response channel.")
print(f"4. Triage the {largest_backlog_priority} priority backlog first and assign owners to aged unresolved tickets.")
print("5. Monitor response and resolution SLA attainment by priority, channel and category each week.")


## 8. Export datasets and dashboard tables

In [ ]:
tickets.to_csv(OUTPUT_DIR / "customer_support_sample_tickets.csv", index=False)
category_perf.to_csv(OUTPUT_DIR / "category_performance.csv")
channel_perf.to_csv(OUTPUT_DIR / "channel_performance.csv")
repeat_summary.to_csv(OUTPUT_DIR / "repeat_query_summary.csv")
unresolved_backlog.to_csv(OUTPUT_DIR / "unresolved_backlog.csv")
sla.to_csv(OUTPUT_DIR / "illustrative_sla_performance.csv")
print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")


## 9. Conclusion
This analysis demonstrates how support operations can be monitored through ticket volume, response time, resolution time, repeat contacts, backlog and priority-based service targets. For real deployment, connect the notebook to an approved ticket export, confirm SLA definitions, account for business hours and holidays, and validate repeat-contact logic with the support team.

**Portfolio note:** All records, targets and results are synthetic/illustrative and should not be represented as actual company data.